In [22]:
# ============================================================
# Cell 01: Notebook Title and Experiment Rule
# Advanced / Energy-Aware Pruning Experiment
# Thesis: Sustainable Edge Vision
# ============================================================

print("=" * 90)
print("Advanced / Energy-Aware Pruning Experiment")
print("Sustainable Edge Vision: Energy-Efficient Deep Learning for Intelligent Crop Disease Detection")
print("=" * 90)

print("""
IMPORTANT RULE:
This notebook is an additional experimental branch.
It will NOT overwrite the official 11% structured-pruned FP32 result.

Current official best result remains:
- Model: 11% Structured-Pruned FP32 MobileNetV2
- Test accuracy: 98.38%
- Model size: 7.1973 MB
- Average CPU forward latency: 25.6517 ms/image
- Total CPU inference energy: 0.0009343 kWh
- Latency reduction vs baseline: 26.21%
- Energy reduction vs baseline: 27.78%

New goal:
Try advanced pruning to see whether more compression is possible while keeping acceptable accuracy.
""")

Advanced / Energy-Aware Pruning Experiment
Sustainable Edge Vision: Energy-Efficient Deep Learning for Intelligent Crop Disease Detection

IMPORTANT RULE:
This notebook is an additional experimental branch.
It will NOT overwrite the official 11% structured-pruned FP32 result.

Current official best result remains:
- Model: 11% Structured-Pruned FP32 MobileNetV2
- Test accuracy: 98.38%
- Model size: 7.1973 MB
- Average CPU forward latency: 25.6517 ms/image
- Total CPU inference energy: 0.0009343 kWh
- Latency reduction vs baseline: 26.21%
- Energy reduction vs baseline: 27.78%

New goal:
Try advanced pruning to see whether more compression is possible while keeping acceptable accuracy.



In [23]:
# ============================================================
# Cell 02: Install Required Libraries
# Installs Torch-Pruning and THOP for dependency-aware pruning and MAC counting
# ============================================================

!pip -q install torch-pruning thop

print("Installation completed.")

Installation completed.


In [24]:
# ============================================================
# Cell 03: Imports and Reproducibility Setup
# Loads all required Python libraries and sets random seed
# ============================================================

import os
import time
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch_pruning as tp

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)
print("Torch-Pruning version:", getattr(tp, "__version__", "unknown"))

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. This notebook should ideally run on Colab GPU.")

Device: cuda
PyTorch version: 2.11.0+cu128
Torch-Pruning version: 1.6.0
GPU: Tesla T4


In [25]:
# ============================================================
# Cell 04: Google Drive Mount and Project Path Setup
# Sets project paths and creates separate advanced experiment folders
# ============================================================

from google.colab import drive

drive.mount("/content/gdrive_mount")

PROJECT_ROOT = Path("/content/gdrive_mount/MyDrive/Thesis/thesis_project")

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_CSV = DATA_DIR / "train_files.csv"
VAL_CSV = DATA_DIR / "val_files.csv"
TEST_CSV = DATA_DIR / "test_files.csv"
LABEL_MAP_CSV = DATA_DIR / "label_map.csv"

BASELINE_MODEL_PATH = PROJECT_ROOT / "01_baseline_before_optimization/models/mobilenetv2_baseline_best.pth"

ADV_ROOT = PROJECT_ROOT / "02_after_pruning_quantization/advanced_pruning_experiments"
ADV_NOTEBOOK_DIR = ADV_ROOT / "notebooks"
ADV_MODEL_DIR = ADV_ROOT / "models"
ADV_RESULT_DIR = ADV_ROOT / "results"
ADV_TEMP_DIR = ADV_ROOT / "temporary_checkpoints"

for d in [ADV_NOTEBOOK_DIR, ADV_MODEL_DIR, ADV_RESULT_DIR, ADV_TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Advanced experiment root:", ADV_ROOT)
print("Baseline model:", BASELINE_MODEL_PATH)

required_paths = [
    PROJECT_ROOT,
    TRAIN_CSV,
    VAL_CSV,
    TEST_CSV,
    LABEL_MAP_CSV,
    BASELINE_MODEL_PATH,
]

for p in required_paths:
    print(p, "->", p.exists())

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files/folders:\n" + "\n".join(missing))

Drive already mounted at /content/gdrive_mount; to attempt to forcibly remount, call drive.mount("/content/gdrive_mount", force_remount=True).
Project root: /content/gdrive_mount/MyDrive/Thesis/thesis_project
Advanced experiment root: /content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/advanced_pruning_experiments
Baseline model: /content/gdrive_mount/MyDrive/Thesis/thesis_project/01_baseline_before_optimization/models/mobilenetv2_baseline_best.pth
/content/gdrive_mount/MyDrive/Thesis/thesis_project -> True
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/train_files.csv -> True
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/val_files.csv -> True
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/test_files.csv -> True
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/label_map.csv -> True
/content/gdrive_mount/MyDrive/Thesis/thesis_project/01_baseline_before_optimization/models/mobilenetv2_baseline_best.pth -> True


In [26]:
# ============================================================
# Cell 05: Robust Dataset Root Detection + Auto Extract PlantVillage.zip
# Finds PlantVillage image root using relative_path from train CSV
# If images are not extracted, it searches PlantVillage.zip, copies it to /content, and extracts it
# ============================================================

import zipfile
import shutil
from pathlib import Path
import pandas as pd

train_df_preview = pd.read_csv(TRAIN_CSV)
first_rel_path = train_df_preview.iloc[0]["relative_path"]
first_file_name = Path(first_rel_path).name
rel_depth = len(Path(first_rel_path).parts)

print("First relative path from CSV:")
print(first_rel_path)
print("First file name:", first_file_name)
print("Relative path depth:", rel_depth)

# ------------------------------------------------------------
# 1. Helper function: check whether a root contains the first CSV image
# ------------------------------------------------------------

def root_has_image(root, rel_path):
    root = Path(root)
    return (root / rel_path).exists()

def infer_root_from_found_file(found_file, rel_path):
    """
    If found_file is:
    /content/PlantVillage/Tomato___.../image.JPG

    and rel_path is:
    Tomato___.../image.JPG

    then root should be:
    /content/PlantVillage
    """
    root = Path(found_file)
    for _ in range(len(Path(rel_path).parts)):
        root = root.parent
    return root

# ------------------------------------------------------------
# 2. First try common extracted dataset roots
# ------------------------------------------------------------

candidate_dataset_roots = [
    DATA_DIR / "PlantVillage",
    PROJECT_ROOT / "data" / "PlantVillage",
    PROJECT_ROOT / "PlantVillage",
    Path("/content/PlantVillage"),
    Path("/content/PlantVillage_extracted"),
    Path("/content/PlantVillage_extracted/PlantVillage"),
    Path("/content/PlantVillage_extracted/color"),
    Path("/content/data/PlantVillage"),
]

DATASET_ROOT = None

print("\nChecking common dataset root locations...")
for root in candidate_dataset_roots:
    exists = root.exists()
    has_img = root_has_image(root, first_rel_path) if exists else False
    print(f"{root} | exists={exists} | sample_image={has_img}")

    if has_img:
        DATASET_ROOT = root
        break

# ------------------------------------------------------------
# 3. If not found, search extracted folders in /content
# ------------------------------------------------------------

if DATASET_ROOT is None:
    print("\nDataset not found in common roots. Searching /content for sample image...")

    for found_file in Path("/content").rglob(first_file_name):
        if str(found_file).endswith(first_rel_path):
            DATASET_ROOT = infer_root_from_found_file(found_file, first_rel_path)
            print("Found sample image at:", found_file)
            break

# ------------------------------------------------------------
# 4. If still not found, search for PlantVillage.zip
# ------------------------------------------------------------

if DATASET_ROOT is None:
    print("\nDataset images not found. Searching for PlantVillage.zip...")

    zip_candidates = [
        DATA_DIR / "PlantVillage.zip",
        PROJECT_ROOT / "PlantVillage.zip",
        PROJECT_ROOT / "data" / "PlantVillage.zip",
        Path("/content/gdrive_mount/MyDrive/PlantVillage.zip"),
        Path("/content/gdrive_mount/MyDrive/Thesis/PlantVillage.zip"),
        Path("/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/PlantVillage.zip"),
    ]

    found_zip = None

    # Check direct candidates first
    for z in zip_candidates:
        print("Checking zip:", z, "->", z.exists())
        if z.exists():
            found_zip = z
            break

    # If not found, search inside project root and MyDrive
    if found_zip is None:
        search_bases = [
            PROJECT_ROOT,
            Path("/content/gdrive_mount/MyDrive"),
        ]

        for base in search_bases:
            if base.exists():
                print("Searching zip under:", base)
                matches = list(base.rglob("PlantVillage*.zip"))
                if len(matches) > 0:
                    found_zip = matches[0]
                    print("Found zip:", found_zip)
                    break

    if found_zip is None:
        raise FileNotFoundError(
            "PlantVillage.zip was not found. Please upload/copy PlantVillage.zip to Google Drive, "
            "preferably inside thesis_project/data/PlantVillage.zip"
        )

    # ------------------------------------------------------------
    # 5. Copy zip to /content and extract
    # ------------------------------------------------------------

    local_zip_path = Path("/content/PlantVillage.zip")
    extract_dir = Path("/content/PlantVillage_extracted")

    print("\nFound PlantVillage zip:")
    print(found_zip)

    if not local_zip_path.exists():
        print("\nCopying zip to local Colab storage. This may take some time...")
        shutil.copy2(found_zip, local_zip_path)
        print("Copied to:", local_zip_path)
    else:
        print("Local zip already exists:", local_zip_path)

    extract_dir.mkdir(parents=True, exist_ok=True)

    print("\nExtracting PlantVillage.zip to:")
    print(extract_dir)

    with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print("Extraction completed.")

    # ------------------------------------------------------------
    # 6. Detect dataset root after extraction
    # ------------------------------------------------------------

    post_extract_candidates = [
        extract_dir,
        extract_dir / "PlantVillage",
        extract_dir / "color",
        Path("/content/PlantVillage"),
    ]

    for root in post_extract_candidates:
        if root_has_image(root, first_rel_path):
            DATASET_ROOT = root
            break

    if DATASET_ROOT is None:
        print("\nSearching extracted folder for sample image...")
        for found_file in extract_dir.rglob(first_file_name):
            if str(found_file).endswith(first_rel_path):
                DATASET_ROOT = infer_root_from_found_file(found_file, first_rel_path)
                print("Found sample image at:", found_file)
                break

# ------------------------------------------------------------
# 7. Final check
# ------------------------------------------------------------

if DATASET_ROOT is None:
    raise FileNotFoundError("PlantVillage dataset root still could not be detected after extraction.")

sample_path = DATASET_ROOT / first_rel_path

print("\n" + "=" * 80)
print("DATASET ROOT DETECTED SUCCESSFULLY")
print("=" * 80)
print("DATASET_ROOT:", DATASET_ROOT)
print("Sample path:", sample_path)
print("Sample exists:", sample_path.exists())
print("=" * 80)

First relative path from CSV:
Tomato___Tomato_Yellow_Leaf_Curl_Virus/0fada544-d797-4168-bbf4-4f9dcfd24ec8___YLCV_NREC 1993.JPG
First file name: 0fada544-d797-4168-bbf4-4f9dcfd24ec8___YLCV_NREC 1993.JPG
Relative path depth: 2

Checking common dataset root locations...
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/PlantVillage | exists=False | sample_image=False
/content/gdrive_mount/MyDrive/Thesis/thesis_project/data/PlantVillage | exists=False | sample_image=False
/content/gdrive_mount/MyDrive/Thesis/thesis_project/PlantVillage | exists=False | sample_image=False
/content/PlantVillage | exists=False | sample_image=False
/content/PlantVillage_extracted | exists=True | sample_image=False
/content/PlantVillage_extracted/PlantVillage | exists=True | sample_image=True

DATASET ROOT DETECTED SUCCESSFULLY
DATASET_ROOT: /content/PlantVillage_extracted/PlantVillage
Sample path: /content/PlantVillage_extracted/PlantVillage/Tomato___Tomato_Yellow_Leaf_Curl_Virus/0fada544-d797-4168-bbf4

In [27]:
# ============================================================
# Cell 06: Load CSV Files and Define Image Transforms
# Uses the same preprocessing as baseline and previous pruning work
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
label_map_df = pd.read_csv(LABEL_MAP_CSV).sort_values("label").reset_index(drop=True)

CLASS_NAMES = label_map_df["class_name"].tolist()
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = 224

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Number of classes:", NUM_CLASSES)

Train images: 40728
Validation images: 8146
Test images: 5431
Number of classes: 38


In [28]:
# ============================================================
# Cell 07: Dataset Class and DataLoaders
# Uses relative_path and label columns from the fixed CSV splits
# ============================================================

class PlantVillageCSVDataset(Dataset):
    def __init__(self, dataframe, dataset_root, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.dataset_root / row["relative_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

BATCH_SIZE_TRAIN = 32
BATCH_SIZE_EVAL = 32
NUM_WORKERS = 2

train_dataset = PlantVillageCSVDataset(train_df, DATASET_ROOT, train_transform)
val_dataset = PlantVillageCSVDataset(val_df, DATASET_ROOT, eval_transform)
test_dataset = PlantVillageCSVDataset(test_df, DATASET_ROOT, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1273
Validation batches: 255
Test batches: 170


In [29]:
# ============================================================
# Cell 08: Load Baseline MobileNetV2 Teacher Model
# Loads the official before-optimization best checkpoint
# ============================================================

def build_mobilenetv2(num_classes):
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

teacher_model = build_mobilenetv2(NUM_CLASSES)

checkpoint = torch.load(
    BASELINE_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    teacher_model.load_state_dict(checkpoint["model_state_dict"])
else:
    teacher_model.load_state_dict(checkpoint)

teacher_model = teacher_model.to(DEVICE)
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

baseline_checkpoint_epoch = checkpoint.get("epoch", None) if isinstance(checkpoint, dict) else None
baseline_best_val_acc = checkpoint.get("best_val_accuracy", None) if isinstance(checkpoint, dict) else None

print("Teacher baseline loaded.")
print("Checkpoint epoch:", baseline_checkpoint_epoch)
print("Best validation accuracy:", baseline_best_val_acc)

Teacher baseline loaded.
Checkpoint epoch: 9
Best validation accuracy: 0.9969310090842131


In [30]:
# ============================================================
# Cell 09: Evaluation Function
# Computes accuracy, macro precision/recall/F1, and weighted F1
# ============================================================

def evaluate_model(model, data_loader, device, desc="Evaluating"):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.inference_mode():
        for images, labels in tqdm(data_loader, desc=desc, leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = accuracy_score(all_labels, all_preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0,
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0,
    )

    avg_loss = total_loss / len(all_labels)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

baseline_val_metrics = evaluate_model(
    teacher_model,
    val_loader,
    DEVICE,
    desc="Baseline teacher validation sanity check",
)

print("Baseline validation sanity check:")
for k, v in baseline_val_metrics.items():
    print(k, ":", v)

Baseline teacher validation sanity check:   0%|          | 0/255 [00:00<?, ?it/s]

Baseline validation sanity check:
loss : 0.011090856459326292
accuracy : 0.9969310090842131
precision_macro : 0.996324167721049
recall_macro : 0.9967992163745207
f1_macro : 0.9965513617249061
precision_weighted : 0.9969549892789684
recall_weighted : 0.9969310090842131
f1_weighted : 0.9969322289388096


In [31]:
# ============================================================
# Cell 10: Model Complexity Utilities
# Counts parameters, MACs, and saved model size
# ============================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_macs_and_params(model, device):
    model.eval()
    example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

    try:
        macs, params = tp.utils.count_ops_and_params(model, example_inputs)
        return float(macs), float(params)
    except Exception as e:
        print("MAC counting failed:", repr(e))
        return None, float(count_parameters(model))

def save_full_model_and_get_size(model, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    current_device = next(model.parameters()).device
    model_cpu = model.to("cpu")
    torch.save(model_cpu, save_path)
    size_mb = save_path.stat().st_size / (1024 * 1024)
    model.to(current_device)

    return size_mb

baseline_macs, baseline_params_from_tp = count_macs_and_params(teacher_model, DEVICE)
baseline_params = count_parameters(teacher_model)
baseline_model_size_mb = BASELINE_MODEL_PATH.stat().st_size / (1024 * 1024)

print("Baseline parameters:", baseline_params)
print("Baseline MACs:", baseline_macs)
print("Baseline model size MB:", baseline_model_size_mb)

Baseline parameters: 2272550
Baseline MACs: 319004966.0
Baseline model size MB: 26.383463859558105


In [32]:
# ============================================================
# Cell 11: Knowledge Distillation Loss
# Uses baseline MobileNetV2 as teacher and pruned model as student
# ============================================================

class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=4.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.kl_loss = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, labels, teacher_logits):
        ce = self.ce_loss(student_logits, labels)

        T = self.temperature

        kd = self.kl_loss(
            F.log_softmax(student_logits / T, dim=1),
            F.softmax(teacher_logits / T, dim=1),
        ) * (T * T)

        total = (1.0 - self.alpha) * ce + self.alpha * kd
        return total, ce.detach(), kd.detach()

KD_ALPHA = 0.5
KD_TEMPERATURE = 4.0

distill_criterion = DistillationLoss(
    alpha=KD_ALPHA,
    temperature=KD_TEMPERATURE,
    label_smoothing=0.05,
)

print("Distillation loss ready.")
print("Alpha:", KD_ALPHA)
print("Temperature:", KD_TEMPERATURE)

Distillation loss ready.
Alpha: 0.5
Temperature: 4.0


In [33]:
# ============================================================
# Cell 12: Improved Safe Sensitivity-Aware Dependency Pruning
# Strategy:
# - Start from current 11% pruned model later
# - Protect sensitive layers
# - Avoid depthwise convolutions as primary pruning targets
# - Prefer pointwise 1x1 convolutions in middle feature blocks
# ============================================================

def set_model_trainable(model, trainable=True):
    for p in model.parameters():
        p.requires_grad = trainable
    return model

def count_trainable_parameters_simple(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def create_student_from_teacher(teacher_model):
    student = copy.deepcopy(teacher_model)
    student = set_model_trainable(student, True)
    student = student.to(DEVICE)
    student.train()

    print("Student model created from teacher.")
    print("Total parameters:", count_parameters(student))
    print("Trainable parameters:", count_trainable_parameters_simple(student))

    if count_trainable_parameters_simple(student) == 0:
        raise RuntimeError("Student has 0 trainable parameters.")

    return student

def create_student_from_any_base(base_model, base_name="base"):
    student = copy.deepcopy(base_model)
    student = set_model_trainable(student, True)
    student = student.to(DEVICE)
    student.train()

    print(f"Student model created from {base_name}.")
    print("Total parameters:", count_parameters(student))
    print("Trainable parameters:", count_trainable_parameters_simple(student))

    if count_trainable_parameters_simple(student) == 0:
        raise RuntimeError("Student has 0 trainable parameters.")

    return student

def is_depthwise_conv(module):
    return (
        isinstance(module, nn.Conv2d)
        and module.groups == module.in_channels
        and module.in_channels == module.out_channels
    )

def is_pointwise_conv(module):
    return (
        isinstance(module, nn.Conv2d)
        and module.kernel_size == (1, 1)
        and module.groups == 1
    )

def build_safe_ignored_layers_for_mobilenetv2(model):
    """
    Build ignored layers for safer MobileNetV2 pruning.

    We ignore:
    - final classifier
    - first stem convolution
    - final feature convolution
    - depthwise convolutions
    - non-pointwise convolutions
    - very small pointwise layers

    This makes pruning focus mostly on parameter-heavy pointwise 1x1 convolutions.
    """
    ignored_layers = []
    target_layers = []

    named_modules = dict(model.named_modules())

    for name, module in model.named_modules():

        # Always protect classifier output
        if isinstance(module, nn.Linear):
            ignored_layers.append(module)
            continue

        if isinstance(module, nn.Conv2d):

            # Protect depthwise convolutions
            if is_depthwise_conv(module):
                ignored_layers.append(module)
                continue

            # Protect first stem conv
            if name.startswith("features.0"):
                ignored_layers.append(module)
                continue

            # Protect final feature conv
            if name.startswith("features.18"):
                ignored_layers.append(module)
                continue

            # Protect non-pointwise convs
            if not is_pointwise_conv(module):
                ignored_layers.append(module)
                continue

            # Protect very small layers
            if module.out_channels < 64:
                ignored_layers.append(module)
                continue

            # Target mostly middle pointwise convs
            target_layers.append((name, module))

    print("\nSafe pruning layer selection:")
    print("Target pointwise layers:", len(target_layers))
    print("Ignored/protected layers:", len(ignored_layers))

    print("\nTarget layers:")
    for name, module in target_layers[:30]:
        print(f"  {name}: in={module.in_channels}, out={module.out_channels}, k={module.kernel_size}")

    if len(target_layers) == 0:
        raise RuntimeError("No target pointwise layers found. Check MobileNetV2 structure.")

    return ignored_layers, target_layers

def apply_safe_pointwise_dependency_pruning(model, pruning_ratio):
    """
    Safer dependency-aware pruning:
    - Protects sensitive/depthwise/first/last/classifier layers
    - Prunes mostly pointwise 1x1 convs
    - Uses Torch-Pruning dependency graph
    """
    model = set_model_trainable(model, True)
    model.eval()

    before_params = count_parameters(model)
    before_macs, _ = count_macs_and_params(model, DEVICE)

    example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    ignored_layers, target_layers = build_safe_ignored_layers_for_mobilenetv2(model)

    importance = tp.importance.MagnitudeImportance(p=2)

    pruner = tp.pruner.MagnitudePruner(
        model,
        example_inputs,
        importance=importance,
        pruning_ratio=pruning_ratio,
        global_pruning=False,
        ignored_layers=ignored_layers,
        round_to=8,
    )

    pruner.step()

    model = set_model_trainable(model, True)
    model = model.to(DEVICE)
    model.train()

    after_params = count_parameters(model)
    after_macs, _ = count_macs_and_params(model, DEVICE)

    print("\nSafe pointwise pruning check:")
    print("Requested pruning ratio:", pruning_ratio)
    print("Before params:", before_params)
    print("After params:", after_params)
    print("Before MACs:", before_macs)
    print("After MACs:", after_macs)

    param_reduction = ((before_params - after_params) / before_params) * 100
    mac_reduction = ((before_macs - after_macs) / before_macs) * 100 if before_macs else None

    print("Parameter reduction %:", param_reduction)
    print("MAC reduction %:", mac_reduction)

    if after_params >= before_params:
        raise RuntimeError(
            "Safe pruning did not reduce parameters. Need different target-layer strategy."
        )

    return model

print("Improved safe sensitivity-aware pruning functions ready.")

Improved safe sensitivity-aware pruning functions ready.


In [34]:
# ============================================================
# Cell 13: Fixed Fine-Tuning Function with Knowledge Distillation
# Fix: ensures student model has trainable parameters before training
# ============================================================

def fine_tune_with_distillation(
    student_model,
    teacher_model,
    train_loader,
    val_loader,
    device,
    epochs=5,
    learning_rate=5e-5,
    weight_decay=1e-4,
    candidate_name="candidate",
):
    student_model = set_model_trainable(student_model, True)
    student_model = student_model.to(device)
    teacher_model = teacher_model.to(device)
    teacher_model.eval()

    trainable_params = count_trainable_parameters_simple(student_model)
    print("Trainable parameters before fine-tuning:", trainable_params)

    if trainable_params == 0:
        raise RuntimeError("Student model has 0 trainable parameters before fine-tuning.")

    optimizer = torch.optim.AdamW(
        [p for p in student_model.parameters() if p.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    history = []
    best_val_acc = -1.0
    best_state_dict = None
    best_epoch = None

    for epoch in range(1, epochs + 1):
        student_model.train()

        running_loss = 0.0
        running_ce = 0.0
        running_kd = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, desc=f"{candidate_name} | Epoch {epoch}/{epochs}", leave=False)

        for images, labels in loop:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.inference_mode():
                teacher_logits = teacher_model(images)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                student_logits = student_model(images)
                loss, ce_part, kd_part = distill_criterion(
                    student_logits,
                    labels,
                    teacher_logits,
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = student_logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            running_loss += loss.item() * labels.size(0)
            running_ce += ce_part.item() * labels.size(0)
            running_kd += kd_part.item() * labels.size(0)

            loop.set_postfix({
                "loss": running_loss / max(total, 1),
                "acc": correct / max(total, 1),
            })

        scheduler.step()

        train_loss = running_loss / total
        train_ce = running_ce / total
        train_kd = running_kd / total
        train_acc = correct / total

        val_metrics = evaluate_model(
            student_model,
            val_loader,
            device,
            desc=f"{candidate_name} validation epoch {epoch}",
        )

        row = {
            "candidate": candidate_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ce": train_ce,
            "train_kd": train_kd,
            "train_accuracy": train_acc,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision_macro": val_metrics["precision_macro"],
            "val_recall_macro": val_metrics["recall_macro"],
            "val_f1_macro": val_metrics["f1_macro"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        history.append(row)

        print(
            f"{candidate_name} | Epoch {epoch}: "
            f"train_acc={train_acc:.4f}, "
            f"val_acc={val_metrics['accuracy']:.4f}, "
            f"val_f1={val_metrics['f1_macro']:.4f}"
        )

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_epoch = epoch
            best_state_dict = copy.deepcopy(student_model.state_dict())

    if best_state_dict is not None:
        student_model.load_state_dict(best_state_dict)

    return student_model, pd.DataFrame(history), best_epoch, best_val_acc

print("Fixed fine-tuning with distillation function ready.")

Fixed fine-tuning with distillation function ready.


In [35]:
# ============================================================
# Cell 14: Colab CPU Latency Proxy Function
# This is NOT final Dell CPU latency.
# It is only used for energy-aware candidate filtering in Colab.
# ============================================================

def quick_colab_cpu_latency_proxy(model, data_loader, max_images=128):
    original_device = next(model.parameters()).device

    model_cpu = model.to("cpu")
    model_cpu.eval()

    latencies = []
    total_images = 0

    # Warm-up small number
    warm_count = 0
    with torch.inference_mode():
        for images, labels in data_loader:
            _ = model_cpu(images.float())
            warm_count += images.size(0)
            if warm_count >= 16:
                break

    with torch.inference_mode():
        for images, labels in data_loader:
            start = time.perf_counter()
            _ = model_cpu(images.float())
            end = time.perf_counter()

            batch_latency = end - start
            per_image_latency = batch_latency / images.size(0)

            latencies.extend([per_image_latency] * images.size(0))
            total_images += images.size(0)

            if total_images >= max_images:
                break

    model.to(original_device)

    latencies = np.array(latencies)

    return {
        "colab_cpu_proxy_images": int(total_images),
        "colab_cpu_proxy_avg_latency_ms": float(np.mean(latencies) * 1000),
        "colab_cpu_proxy_median_latency_ms": float(np.median(latencies) * 1000),
    }

print("Colab CPU latency proxy function ready.")
print("Reminder: This is only a proxy, not final Dell CPU measurement.")

Colab CPU latency proxy function ready.
Reminder: This is only a proxy, not final Dell CPU measurement.


In [36]:
# ============================================================
# Cell 15: Candidate Evaluation and Saving Function
# Saves candidate model and summary without overwriting official results
# ============================================================

def evaluate_and_save_candidate(
    model,
    pruning_ratio,
    history_df,
    best_epoch,
    candidate_name,
):
    model.eval()

    val_metrics = evaluate_model(
        model,
        val_loader,
        DEVICE,
        desc=f"Final validation {candidate_name}",
    )

    candidate_macs, candidate_params_tp = count_macs_and_params(model, DEVICE)
    candidate_params = count_parameters(model)
    candidate_trainable_params = count_trainable_parameters(model)

    save_path = ADV_MODEL_DIR / f"{candidate_name}_full_model.pth"
    model_size_mb = save_full_model_and_get_size(model, save_path)

    # Bring model back to device after saving.
    model.to(DEVICE)

    latency_proxy = quick_colab_cpu_latency_proxy(
        model,
        val_loader,
        max_images=128,
    )

    param_reduction_percent = ((baseline_params - candidate_params) / baseline_params) * 100

    if baseline_macs is not None and candidate_macs is not None:
        mac_reduction_percent = ((baseline_macs - candidate_macs) / baseline_macs) * 100
    else:
        mac_reduction_percent = None

    size_reduction_percent = ((baseline_model_size_mb - model_size_mb) / baseline_model_size_mb) * 100

    val_acc_drop_pp = (baseline_val_metrics["accuracy"] - val_metrics["accuracy"]) * 100
    val_f1_drop_pp = (baseline_val_metrics["f1_macro"] - val_metrics["f1_macro"]) * 100

    # Energy-aware proxy score:
    # This is not a new algorithmic claim. It is a practical candidate selection score.
    # Higher is better.
    if mac_reduction_percent is None:
        mac_component = 0
    else:
        mac_component = mac_reduction_percent

    energy_aware_proxy_score = (
        (val_metrics["accuracy"] * 100) * 0.65
        + mac_component * 0.15
        + size_reduction_percent * 0.10
        + param_reduction_percent * 0.10
        - max(val_acc_drop_pp - 1.50, 0) * 5.0
    )

    summary = {
        "candidate_name": candidate_name,
        "pruning_ratio_requested": pruning_ratio,
        "best_epoch": best_epoch,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_accuracy_percent": val_metrics["accuracy"] * 100,
        "val_precision_macro": val_metrics["precision_macro"],
        "val_recall_macro": val_metrics["recall_macro"],
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_macro_percent": val_metrics["f1_macro"] * 100,
        "val_acc_drop_pp_vs_baseline": val_acc_drop_pp,
        "val_f1_drop_pp_vs_baseline": val_f1_drop_pp,
        "parameters": candidate_params,
        "trainable_parameters": candidate_trainable_params,
        "baseline_parameters": baseline_params,
        "parameter_reduction_percent": param_reduction_percent,
        "macs": candidate_macs,
        "baseline_macs": baseline_macs,
        "mac_reduction_percent": mac_reduction_percent,
        "model_size_mb": model_size_mb,
        "baseline_model_size_mb": baseline_model_size_mb,
        "model_size_reduction_percent": size_reduction_percent,
        "model_path": str(save_path),
        "energy_aware_proxy_score": energy_aware_proxy_score,
        **latency_proxy,
        "note": (
            "Advanced pruning candidate. Colab CPU latency is only a proxy. "
            "Final latency and energy must be measured on Dell Latitude CPU using matched process."
        ),
    }

    # Save history for this candidate
    history_path = ADV_RESULT_DIR / f"{candidate_name}_training_history.csv"
    history_df.to_csv(history_path, index=False)

    # Save summary for this candidate
    summary_path = ADV_RESULT_DIR / f"{candidate_name}_summary.csv"
    pd.DataFrame([summary]).to_csv(summary_path, index=False)

    print(f"Saved model: {save_path}")
    print(f"Saved history: {history_path}")
    print(f"Saved summary: {summary_path}")

    return summary

print("Candidate evaluation and saving function ready.")

Candidate evaluation and saving function ready.


In [37]:
# ============================================================
# Cell 16A: Load Current Official 11% Pruned FP32 Model as Base
# This model will be used for incremental advanced pruning.
# It will NOT be overwritten.
# ============================================================

CURRENT_11P_MODEL_PATH = PROJECT_ROOT / "02_after_pruning_quantization/models/mobilenetv2_pruned_best_full_model.pth"

if not CURRENT_11P_MODEL_PATH.exists():
    raise FileNotFoundError(f"Current 11% pruned model not found: {CURRENT_11P_MODEL_PATH}")

print("Loading current official 11% pruned model:")
print(CURRENT_11P_MODEL_PATH)

try:
    loaded_obj = torch.load(CURRENT_11P_MODEL_PATH, map_location="cpu", weights_only=False)
except TypeError:
    loaded_obj = torch.load(CURRENT_11P_MODEL_PATH, map_location="cpu")

if isinstance(loaded_obj, torch.nn.Module):
    pruned_base_model = loaded_obj
elif isinstance(loaded_obj, dict) and "model" in loaded_obj:
    pruned_base_model = loaded_obj["model"]
else:
    raise RuntimeError(f"Unsupported model format: {type(loaded_obj)}")

pruned_base_model = pruned_base_model.to(DEVICE)
pruned_base_model.eval()

for p in pruned_base_model.parameters():
    p.requires_grad = False

current_11p_val_metrics = evaluate_model(
    pruned_base_model,
    val_loader,
    DEVICE,
    desc="Current 11% pruned base validation check",
)

current_11p_macs, _ = count_macs_and_params(pruned_base_model, DEVICE)
current_11p_params = count_parameters(pruned_base_model)
current_11p_size_mb = CURRENT_11P_MODEL_PATH.stat().st_size / (1024 * 1024)

print("\nCurrent official 11% pruned base:")
print("Val accuracy:", current_11p_val_metrics["accuracy"] * 100)
print("Val macro F1:", current_11p_val_metrics["f1_macro"] * 100)
print("Parameters:", current_11p_params)
print("MACs:", current_11p_macs)
print("Model size MB:", current_11p_size_mb)

Loading current official 11% pruned model:
/content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/models/mobilenetv2_pruned_best_full_model.pth


Current 11% pruned base validation check:   0%|          | 0/255 [00:00<?, ?it/s]


Current official 11% pruned base:
Val accuracy: 98.62509206972747
Val macro F1: 98.26975305474518
Parameters: 1805093
MACs: 254701367.0
Model size MB: 7.1973371505737305


In [38]:
# ============================================================
# Cell 16B: Create Student from Current 11% Pruned Base
# Further pruning starts from the already selected 11% pruned model.
# ============================================================

def create_student_from_current_11p_base(pruned_base_model):
    student = create_student_from_any_base(
        pruned_base_model,
        base_name="current official 11% pruned model"
    )
    return student

print("Current 11% base student function ready.")

Current 11% base student function ready.


In [39]:
# ============================================================
# Cell 16C: Evaluate and Save Incremental Advanced Candidate
# Adds comparison vs both baseline and current official 11% pruned model
# ============================================================

def evaluate_and_save_incremental_candidate(
    model,
    extra_pruning_ratio,
    history_df,
    best_epoch,
    candidate_name,
):
    model.eval()

    val_metrics = evaluate_model(
        model,
        val_loader,
        DEVICE,
        desc=f"Final validation {candidate_name}",
    )

    candidate_macs, _ = count_macs_and_params(model, DEVICE)
    candidate_params = count_parameters(model)
    candidate_trainable_params = count_trainable_parameters(model)

    save_path = ADV_MODEL_DIR / f"{candidate_name}_full_model_NOT_FINAL.pth"
    model_size_mb = save_full_model_and_get_size(model, save_path)

    model.to(DEVICE)

    latency_proxy = quick_colab_cpu_latency_proxy(
        model,
        val_loader,
        max_images=128,
    )

    param_reduction_vs_baseline = ((baseline_params - candidate_params) / baseline_params) * 100
    mac_reduction_vs_baseline = ((baseline_macs - candidate_macs) / baseline_macs) * 100
    size_reduction_vs_baseline = ((baseline_model_size_mb - model_size_mb) / baseline_model_size_mb) * 100

    val_acc_drop_pp_vs_baseline = (baseline_val_metrics["accuracy"] - val_metrics["accuracy"]) * 100
    val_f1_drop_pp_vs_baseline = (baseline_val_metrics["f1_macro"] - val_metrics["f1_macro"]) * 100

    param_reduction_vs_current_11p = ((current_11p_params - candidate_params) / current_11p_params) * 100
    mac_reduction_vs_current_11p = ((current_11p_macs - candidate_macs) / current_11p_macs) * 100
    size_reduction_vs_current_11p = ((current_11p_size_mb - model_size_mb) / current_11p_size_mb) * 100

    val_acc_drop_pp_vs_current_11p = (current_11p_val_metrics["accuracy"] - val_metrics["accuracy"]) * 100
    val_f1_drop_pp_vs_current_11p = (current_11p_val_metrics["f1_macro"] - val_metrics["f1_macro"]) * 100

    energy_aware_proxy_score = (
        (val_metrics["accuracy"] * 100) * 0.72
        + mac_reduction_vs_baseline * 0.10
        + size_reduction_vs_baseline * 0.08
        + param_reduction_vs_baseline * 0.10
        - max(val_acc_drop_pp_vs_current_11p - 0.30, 0) * 10.0
    )

    summary = {
        "candidate_name": candidate_name,
        "base_model": "current_official_11_percent_pruned_fp32",
        "extra_pruning_ratio_requested": extra_pruning_ratio,
        "best_epoch": best_epoch,

        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_accuracy_percent": val_metrics["accuracy"] * 100,
        "val_f1_macro": val_metrics["f1_macro"],
        "val_f1_macro_percent": val_metrics["f1_macro"] * 100,

        "parameters": candidate_params,
        "trainable_parameters": candidate_trainable_params,
        "macs": candidate_macs,
        "model_size_mb": model_size_mb,
        "model_path": str(save_path),

        "baseline_parameters": baseline_params,
        "baseline_macs": baseline_macs,
        "baseline_model_size_mb": baseline_model_size_mb,
        "param_reduction_percent_vs_baseline": param_reduction_vs_baseline,
        "mac_reduction_percent_vs_baseline": mac_reduction_vs_baseline,
        "model_size_reduction_percent_vs_baseline": size_reduction_vs_baseline,
        "val_acc_drop_pp_vs_baseline": val_acc_drop_pp_vs_baseline,
        "val_f1_drop_pp_vs_baseline": val_f1_drop_pp_vs_baseline,

        "current_11p_parameters": current_11p_params,
        "current_11p_macs": current_11p_macs,
        "current_11p_model_size_mb": current_11p_size_mb,
        "param_reduction_percent_vs_current_11p": param_reduction_vs_current_11p,
        "mac_reduction_percent_vs_current_11p": mac_reduction_vs_current_11p,
        "model_size_reduction_percent_vs_current_11p": size_reduction_vs_current_11p,
        "val_acc_drop_pp_vs_current_11p": val_acc_drop_pp_vs_current_11p,
        "val_f1_drop_pp_vs_current_11p": val_f1_drop_pp_vs_current_11p,

        "energy_aware_proxy_score": energy_aware_proxy_score,
        **latency_proxy,

        "note": (
            "Safe sensitivity-aware incremental pruning candidate from current official 11% pruned FP32 model. "
            "This is NOT FINAL. Final decision requires Dell CPU test accuracy and matched latency-energy measurement."
        ),
    }

    history_path = ADV_RESULT_DIR / f"{candidate_name}_training_history.csv"
    history_df.to_csv(history_path, index=False)

    summary_path = ADV_RESULT_DIR / f"{candidate_name}_summary.csv"
    pd.DataFrame([summary]).to_csv(summary_path, index=False)

    print("Saved model:", save_path)
    print("Saved history:", history_path)
    print("Saved summary:", summary_path)

    return summary

print("Incremental candidate saving function ready.")

Incremental candidate saving function ready.


In [40]:
# ============================================================
# Cell 16D: Safe Sensitivity-Aware Incremental Pruning from Current 11%
# First run only extra 2%
# ============================================================

EXTRA_PRUNING_RATIOS = [0.02]

EPOCHS_PER_CANDIDATE = 10
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-4

incremental_candidate_summaries = []

for extra_ratio in EXTRA_PRUNING_RATIOS:
    candidate_name = f"safe_pointwise_kd_from_11p_extra_{int(extra_ratio * 100)}percent"

    print("\n" + "=" * 90)
    print("Starting safe incremental candidate:", candidate_name)
    print("=" * 90)

    student_model = create_student_from_current_11p_base(pruned_base_model)

    student_model = apply_safe_pointwise_dependency_pruning(
        student_model,
        pruning_ratio=extra_ratio,
    )

    pruned_macs, _ = count_macs_and_params(student_model, DEVICE)
    pruned_params = count_parameters(student_model)

    print("After safe incremental pruning before fine-tuning:")
    print("Parameters:", pruned_params)
    print("MACs:", pruned_macs)

    student_model, history_df, best_epoch, best_val_acc = fine_tune_with_distillation(
        student_model=student_model,
        teacher_model=teacher_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        epochs=EPOCHS_PER_CANDIDATE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        candidate_name=candidate_name,
    )

    summary = evaluate_and_save_incremental_candidate(
        model=student_model,
        extra_pruning_ratio=extra_ratio,
        history_df=history_df,
        best_epoch=best_epoch,
        candidate_name=candidate_name,
    )

    incremental_candidate_summaries.append(summary)

    incremental_comparison_df = pd.DataFrame(incremental_candidate_summaries)
    incremental_comparison_path = ADV_RESULT_DIR / "safe_incremental_pruning_from_11p_comparison_running.csv"
    incremental_comparison_df.to_csv(incremental_comparison_path, index=False)

    print("\nCandidate completed:", candidate_name)
    print("Validation accuracy:", summary["val_accuracy_percent"])
    print("Macro F1:", summary["val_f1_macro_percent"])
    print("Model size MB:", summary["model_size_mb"])
    print("Param reduction vs current 11%:", summary["param_reduction_percent_vs_current_11p"])
    print("MAC reduction vs current 11%:", summary["mac_reduction_percent_vs_current_11p"])
    print("Size reduction vs current 11%:", summary["model_size_reduction_percent_vs_current_11p"])
    print("Accuracy drop vs current 11%:", summary["val_acc_drop_pp_vs_current_11p"])
    print("Energy-aware proxy score:", summary["energy_aware_proxy_score"])

    del student_model
    torch.cuda.empty_cache()

print("\nSafe sensitivity-aware incremental pruning first test completed.")


Starting safe incremental candidate: safe_pointwise_kd_from_11p_extra_2percent
Student model created from current official 11% pruned model.
Total parameters: 1805093
Trainable parameters: 1805093

Safe pruning layer selection:
Target pointwise layers: 23
Ignored/protected layers: 30

Target layers:
  features.2.conv.0.0: in=14, out=85, k=(1, 1)
  features.3.conv.0.0: in=21, out=128, k=(1, 1)
  features.4.conv.0.0: in=21, out=128, k=(1, 1)
  features.5.conv.0.0: in=28, out=170, k=(1, 1)
  features.6.conv.0.0: in=28, out=170, k=(1, 1)
  features.7.conv.0.0: in=28, out=170, k=(1, 1)
  features.8.conv.0.0: in=56, out=341, k=(1, 1)
  features.9.conv.0.0: in=56, out=341, k=(1, 1)
  features.10.conv.0.0: in=56, out=341, k=(1, 1)
  features.11.conv.0.0: in=56, out=341, k=(1, 1)
  features.11.conv.2: in=341, out=85, k=(1, 1)
  features.12.conv.0.0: in=85, out=512, k=(1, 1)
  features.12.conv.2: in=512, out=85, k=(1, 1)
  features.13.conv.0.0: in=85, out=512, k=(1, 1)
  features.13.conv.2: in=

safe_pointwise_kd_from_11p_extra_2percent | Epoch 1/10:   0%|          | 0/1273 [00:00<?, ?it/s]

safe_pointwise_kd_from_11p_extra_2percent validation epoch 1:   0%|          | 0/255 [00:00<?, ?it/s]

safe_pointwise_kd_from_11p_extra_2percent | Epoch 1: train_acc=0.9844, val_acc=0.9921, val_f1=0.9906


safe_pointwise_kd_from_11p_extra_2percent | Epoch 2/10:   0%|          | 0/1273 [00:00<?, ?it/s]

...

safe_pointwise_kd_from_11p_extra_2percent | Epoch 10/10:   0%|          | 0/1273 [00:00<?, ?it/s]

safe_pointwise_kd_from_11p_extra_2percent validation epoch 10:   0%|          | 0/255 [00:00<?, ?it/s]

safe_pointwise_kd_from_11p_extra_2percent | Epoch 10: train_acc=0.9969, val_acc=0.9948, val_f1=0.9931


Final validation safe_pointwise_kd_from_11p_extra_2percent:   0%|          | 0/255 [00:00<?, ?it/s]

Saved model: /content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/advanced_pruning_experiments/models/safe_pointwise_kd_from_11p_extra_2percent_full_model_NOT_FINAL.pth
Saved history: /content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/advanced_pruning_experiments/results/safe_pointwise_kd_from_11p_extra_2percent_training_history.csv
Saved summary: /content/gdrive_mount/MyDrive/Thesis/thesis_project/02_after_pruning_quantization/advanced_pruning_experiments/results/safe_pointwise_kd_from_11p_extra_2percent_summary.csv

Candidate completed: safe_pointwise_kd_from_11p_extra_2percent
Validation accuracy: 99.54578934446354
Macro F1: 99.36743637724416
Model size MB: 6.9021711349487305
Param reduction vs current 11%: 3.7229660743241486
MAC reduction vs current 11%: 2.1317482760114124
Size reduction vs current 11%: 4.101044725985513
Accuracy drop vs current 11%: -0.9206972747360709
Energy-aware proxy score: 82.11873733186351

Safe sen

In [41]:
# ============================================================
# Cell 16E: Calculate Actual Total Pruning / Compression
# Run this after Cell 16D completes
# ============================================================

import pandas as pd
from pathlib import Path

# Load latest incremental comparison result
comparison_path = ADV_RESULT_DIR / "safe_incremental_pruning_from_11p_comparison_running.csv"

if comparison_path.exists():
    df = pd.read_csv(comparison_path)
else:
    df = pd.DataFrame(incremental_candidate_summaries)

if len(df) == 0:
    raise RuntimeError("No incremental candidate summary found. Run Cell 16D first.")

# Show all candidates
display_cols = [
    "candidate_name",
    "extra_pruning_ratio_requested",
    "val_accuracy_percent",
    "val_f1_macro_percent",
    "parameters",
    "macs",
    "model_size_mb",
    "param_reduction_percent_vs_baseline",
    "mac_reduction_percent_vs_baseline",
    "model_size_reduction_percent_vs_baseline",
    "param_reduction_percent_vs_current_11p",
    "mac_reduction_percent_vs_current_11p",
    "model_size_reduction_percent_vs_current_11p",
    "val_acc_drop_pp_vs_current_11p",
    "model_path",
]

display(df[display_cols])

# Pick the latest candidate
latest = df.iloc[-1]

print("=" * 90)
print("Actual Total Pruning / Compression Summary")
print("=" * 90)

print("Candidate:", latest["candidate_name"])
print("Requested setup: selected 11% pruned model + extra",
      latest["extra_pruning_ratio_requested"] * 100, "% incremental safe pruning")

print("\nCompared with original baseline MobileNetV2:")
print(f"Baseline parameters: {baseline_params:,}")
print(f"Candidate parameters: {int(latest['parameters']):,}")
print(f"Actual total parameter reduction: {latest['param_reduction_percent_vs_baseline']:.2f}%")

print(f"\nBaseline MACs: {baseline_macs:,.0f}")
print(f"Candidate MACs: {latest['macs']:,.0f}")
print(f"Actual total MAC reduction: {latest['mac_reduction_percent_vs_baseline']:.2f}%")

print(f"\nBaseline model size: {baseline_model_size_mb:.4f} MB")
print(f"Candidate model size: {latest['model_size_mb']:.4f} MB")
print(f"Actual total model-size reduction: {latest['model_size_reduction_percent_vs_baseline']:.2f}%")

print("\nCompared with current official 11% pruned model:")
print(f"Current 11% parameters: {current_11p_params:,}")
print(f"Candidate parameters: {int(latest['parameters']):,}")
print(f"Extra parameter reduction vs current 11%: {latest['param_reduction_percent_vs_current_11p']:.2f}%")

print(f"\nCurrent 11% MACs: {current_11p_macs:,.0f}")
print(f"Candidate MACs: {latest['macs']:,.0f}")
print(f"Extra MAC reduction vs current 11%: {latest['mac_reduction_percent_vs_current_11p']:.2f}%")

print(f"\nCurrent 11% model size: {current_11p_size_mb:.4f} MB")
print(f"Candidate model size: {latest['model_size_mb']:.4f} MB")
print(f"Extra model-size reduction vs current 11%: {latest['model_size_reduction_percent_vs_current_11p']:.2f}%")

print("\nAccuracy:")
print(f"Current 11% validation accuracy: {current_11p_val_metrics['accuracy'] * 100:.4f}%")
print(f"Candidate validation accuracy: {latest['val_accuracy_percent']:.4f}%")
print(f"Accuracy change vs current 11%: {-latest['val_acc_drop_pp_vs_current_11p']:.4f} percentage points")

print("=" * 90)
print("Important reporting note:")
print("Do NOT report total pruning simply as 11% + extra 2% = 13%.")
print("Report the actual parameter, MAC, and model-size reductions shown above.")
print("=" * 90)

,candidate_name,extra_pruning_ratio_requested,val_accuracy_percent,val_f1_macro_percent,parameters,macs,model_size_mb,param_reduction_percent_vs_baseline,mac_reduction_percent_vs_baseline,model_size_reduction_percent_vs_baseline,param_reduction_percent_vs_current_11p,mac_reduction_percent_vs_current_11p,model_size_reduction_percent_vs_current_11p,val_acc_drop_pp_vs_current_11p,model_path
0,safe_pointwise_kd_from_11p_extra_2percent,0.02,99.545789,99.367436,1737890,249271775.0,6.902171,23.526875,21.859594,73.839026,3.722966,2.131748,4.101045,-0.920697,/content/gdrive_mount/MyDrive/Thesis/thesis_pr...


Actual Total Pruning / Compression Summary
Candidate: safe_pointwise_kd_from_11p_extra_2percent
Requested setup: selected 11% pruned model + extra 2.0 % incremental safe pruning

Compared with original baseline MobileNetV2:
Baseline parameters: 2,272,550
Candidate parameters: 1,737,890
Actual total parameter reduction: 23.53%

Baseline MACs: 319,004,966
Candidate MACs: 249,271,775
Actual total MAC reduction: 21.86%

Baseline model size: 26.3835 MB
Candidate model size: 6.9022 MB
Actual total model-size reduction: 73.84%

Compared with current official 11% pruned model:
Current 11% parameters: 1,805,093
Candidate parameters: 1,737,890
Extra parameter reduction vs current 11%: 3.72%

Current 11% MACs: 254,701,367
Candidate MACs: 249,271,775
Extra MAC reduction vs current 11%: 2.13%

Current 11% model size: 7.1973 MB
Candidate model size: 6.9022 MB
Extra model-size reduction vs current 11%: 4.10%

Accuracy:
Current 11% validation accuracy: 98.6251%
Candidate validation accuracy: 99.5458%
